# Download Folder from Workspace

This notebook zips a folder from your workspace, uploads it to a Snowflake stage, and provides instructions to download it to your local machine.

**Steps:**
1. Specify the folder path to download
2. Zip the folder
3. Upload to a Snowflake internal stage
4. Download from stage using SnowSQL or Snowflake CLI locally

In [ ]:
# === CONFIGURE THIS ===
# Set the folder path you want to download (relative to workspace root)
FOLDER_TO_DOWNLOAD = "DQM"  # <-- Change this to your folder name

# Database and schema to use for the stage
DATABASE = "SAMPLE_DATA"  # <-- Change to your database
SCHEMA = "PUBLIC"     # <-- Change to your schema

# Stage name (will be created if it doesn't exist)
STAGE_NAME = "DOWNLOAD_STAGE"

print(f"Folder to download: {FOLDER_TO_DOWNLOAD}")
print(f"Stage: {DATABASE}.{SCHEMA}.{STAGE_NAME}")

In [ ]:
%%sql -r dataframe_1
CREATE DATABASE IF NOT EXISTS {{DATABASE}};
USE DATABASE {{DATABASE}};
CREATE SCHEMA IF NOT EXISTS {{SCHEMA}};

In [ ]:
import os
import zipfile

# Try multiple possible workspace paths (incl. parent of cwd, since the
# notebook may run from *inside* the target folder)
possible_roots = [
    "/workspace",
    os.path.realpath("/workspace"),
    os.getcwd(),
    os.path.dirname(os.getcwd()),
]
folder_path = None

# Case 1: the target folder is one of the roots' children
for root in possible_roots:
    candidate = os.path.join(root, FOLDER_TO_DOWNLOAD)
    if os.path.isdir(candidate):
        folder_path = candidate
        break

# Case 2: the cwd or one of its ancestors IS the target folder
if folder_path is None:
    check = os.path.realpath(os.getcwd())
    while check != os.path.dirname(check):  # stop at filesystem root
        if os.path.basename(check) == FOLDER_TO_DOWNLOAD:
            folder_path = check
            break
        check = os.path.dirname(check)

if folder_path is None:
    cwd = os.getcwd()
    print(f"Current working directory: {cwd}")
    print(f"Contents of cwd:")
    for item in os.listdir(cwd):
        print(f"  {item}")
    raise FileNotFoundError(f"Folder '{FOLDER_TO_DOWNLOAD}' not found. Check the folder name above.")

# Count files
file_count = sum(len(files) for _, _, files in os.walk(folder_path))
print(f"Found folder: {folder_path}")
print(f"Total files: {file_count}")

# Store for later cells
workspace_root = os.path.dirname(folder_path)

# List top-level contents
print("\nContents:")
for item in sorted(os.listdir(folder_path)):
    item_path = os.path.join(folder_path, item)
    if os.path.isdir(item_path):
        print(f"  [DIR]  {item}")
    else:
        print(f"  [FILE] {item}")

In [ ]:
from datetime import datetime

# Zip the folder with datetime in filename
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
zip_filename = f"{FOLDER_TO_DOWNLOAD.replace('/', '_')}_{timestamp}.zip"
zip_path = f"/tmp/{zip_filename}"

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            file_path_full = os.path.join(root, file)
            arcname = os.path.relpath(file_path_full, workspace_root)
            zipf.write(file_path_full, arcname)

zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f"Created zip file: {zip_filename}")
print(f"Zip size: {zip_size_mb:.2f} MB")

In [ ]:
%%sql -r create_stage_result
CREATE OR REPLACE STAGE {{DATABASE}}.{{SCHEMA}}.{{STAGE_NAME}}
  DIRECTORY = (ENABLE = TRUE) 
  ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE');

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

fq_stage = f"{DATABASE}.{SCHEMA}.{STAGE_NAME}"

# Upload zip to stage
put_result = session.sql(f"PUT 'file://{zip_path}' @{fq_stage} AUTO_COMPRESS=FALSE OVERWRITE=TRUE").collect()

print("Upload complete!")
for row in put_result:
    print(f"  File: {row['source']}, Status: {row['status']}")

In [ ]:
# Generate presigned download URL (valid for 1 hour)
result = session.sql(f"SELECT GET_PRESIGNED_URL(@{fq_stage}, '{zip_filename}', 3600) AS url").collect()
download_url = result[0]['URL']

print("="*60)
print("  DOWNLOAD LINK")
print("="*60)
print()
print(f"File: {zip_filename}")
print()
print("Copy and paste this URL in your browser to download:")
print()
print(download_url)
print()
print("(Link expires in 1 hour)")
print("="*60)